# next-gen-aiam — Overview & Reading Guide

This repository is a comparative strategy harness extending Hilpisch's
*Python and AI for Asset Management*.
Every allocation rule — classical mean-variance, regime-switching, factor models,
ML return forecasting, DL sequence models, reinforcement learning, and LLM-augmented views —
implements the same `predict_weights(panel, asof) → weights` interface and is evaluated on a
single harness: 29 assets, 2003–2026, monthly rebalancing, net of 10 bps one-way
transaction costs.

## Two-paper structure

| | Scope | Eval window | Strategy count |
|---|---|---|---|
| **Paper 1** | Deterministic allocation families | Full sample 2003–2026 | 31 base + 31 VMP = 62 |
| **Paper 2** | Learned approaches (ML / DL / RL / LLM) | True OOS 2023–2026 | Each vs. Paper 1 bar |

Paper 1 is the reference layer: deterministic rules cannot overfit (the only modeling choice,
the covariance estimator, is disclosed and principled), so full-sample scoring is
methodologically clean.
Paper 2 restricts evaluation to the post-training window (2023+) and applies epistemic
discounting by paradigm — see **Evaluation discipline** below.

In [ ]:
import sys
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from _shared import (
    ROOT, FAMILY_COLORS, FAMILY_MAP, FAMILY_ORDER, DISPLAY,
    load_base_returns, load_vmp_returns, build_switch_v2a,
    ann_sharpe, ann_return, ann_vol, max_drawdown,
)

matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

base   = load_base_returns()
vmp    = load_vmp_returns()
sw_oos = pd.read_parquet(ROOT / "data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet")
canon  = pd.read_csv(ROOT / "data/cache/appendix_a_canonical.csv")
ret    = pd.read_parquet(ROOT / "data/cache/returns_29_2003_2026.parquet")

switch_v2a = build_switch_v2a(base, sw_oos)

print(f"Universe : {ret.shape[1]} assets x {len(ret)} trading days "
      f"({ret.index[0].date()} to {ret.index[-1].date()}, ~{len(ret)/252:.1f} yrs)")
print(f"Strategies: {base.shape[1]} base + {vmp.shape[1]} VMP variants = {len(canon)} in canonical CSV")

## The universe

**29 assets, 2003-01-03 to 2026-04-30 (~23.3 years)**, spanning three major stress episodes:
GFC (Sep 2008 – Mar 2009), COVID crash (Feb–Apr 2020), and the 2022 rate-shock bear market
(Jan–Oct 2022).

| Asset class | Tickers |
|---|---|
| US single stocks (8) | AAPL, MSFT, GOOGL, NVDA, JPM, JNJ, XOM, WMT |
| US sector ETFs (6) | XLK, XLF, XLE, XLV, XLP, XLU |
| Broad equity ETFs (2) | SPY, IWM |
| International equity ETFs (3) | EFA, EEM, FXI |
| Fixed income (5) | SHY, IEF, TLT, AGG, HYG |
| Commodities & FX (5) | GLD, SLV, DBC, USO, EURUSD |

BTC-USD is excluded entirely for survivorship hygiene (not present for the full 2003–2026 window).
All price data is sourced from EODHD and stored in `data/cache/returns_29_2003_2026.parquet`
(gitignored; regenerate with `scripts/build_returns_29.py`).

## Allocation families and the common harness

### Part 1 — deterministic families (8 families, 31 base strategies)

| Family (exact `FAMILY_MAP` label) | Core strategies | Variants |
|---|---|---|
| Classical MV | EW, GMV, MSR | sample / Ledoit-Wolf / OAS covariance |
| Constrained MV | MSR_C, MVO_C | sample, Ledoit-Wolf |
| Diversification | MDP, RP, HRP | sample, Ledoit-Wolf |
| Regime Switch | SWITCH | sample, Ledoit-Wolf |
| TS Momentum | TSMOM | 12-month, 6-month lookback |
| Black-Litterman | BL-Eq, BL-Mom, BL-Rev | Ledoit-Wolf |
| Factor | FF3-Mom, FF3-LowVol, FF3-Quality, FF3-Multi | — |
| Long-Short | TSMOM-LS, BL-Mom-LS, FF3-Mom-LS | — |

Each of the 31 base strategies is paired with a **VMP overlay** variant (Moreira & Muir 2017):
leverage is adjusted daily to target each strategy's own long-run realized volatility,
using a 21-day lookback, 1-day lag, and a (0.25, 1.5) exposure clip — yielding 62 strategies total.

### Part 2 — learned paradigms

ML (Lasso / RF / XGBoost return forecasting), DL (MLP / LSTM / Transformer), RL (REINFORCE / PPO),
unsupervised representation learning, and LLM-augmented Black-Litterman views.

### The common interface

Every strategy, deterministic or learned, subclasses `PointInTimeStrategy` in `src/aiam/strategy/`:

```
predict_weights(panel: Panel, asof: Timestamp) -> pd.Series
```

`Panel.slice(asof, kind, lookback, freq, fill)` is the no-look-ahead gatekeeper — it enforces that
any data access is strictly prior to `asof`. The harness calls `predict_weights` at each monthly
rebalance date and scores all strategies on the same realized returns.

**Common metrics:** annualized Sharpe, annualized return and volatility, maximum drawdown, Calmar
ratio, monthly hit rate, annualized one-way turnover, and net-of-10bps Sharpe.

## Evaluation discipline & the apples-to-pears caveat

**This is the spine of the two-paper framework.** Part 1 and Part 2 Sharpes are *not* directly
rankable in one column — they measure different things on different windows.

### Full-sample vs. true OOS

| | Part 1 (deterministic) | Part 2 (learned) |
|---|---|---|
| Eval window | Full sample 2003–2026 | True OOS 2023–2026 |
| Justification | No free parameters to overfit; cov estimator choice is disclosed | Trained on 2003–2022; 2023+ is genuinely unseen |
| Key risk | — | 2023–2026 was a favorable period; OOS Sharpes carry a window tailwind |

### Three epistemic tiers

1. **Deterministic** (Paper 1) — exactly reproducible from seed-free rules and public prices.
   Full-sample Sharpes are stable and interpretable.

2. **Learned** (Paper 2, §ML / DL / RL) — exposed to seed variance, hyperparameter choice, and
   overfitting risk. Headline numbers carry error bands; walk-forward OOS discipline applies.

3. **LLM** (Paper 2, §08) — adds training-data contamination risk (model cutoffs fall inside the
   backtest window). Results are labelled *suggestive, not conclusive*; a dedicated hindsight test
   is included in §08.

### Why two papers

A single ranking column would conflate full-sample deterministic Sharpes with 3-year OOS learned
Sharpes on a favorable window — producing a spurious leaderboard. Paper 2 instead measures each
learned paradigm *against the Paper 1 bar*: the ML headline is MSR(Ensemble_mu) OOS Sharpe 2.579;
classical strategies score 0.924–1.372 full-sample. The relevant question for Paper 2 is whether
learning adds genuine skill beyond the best deterministic family on the same OOS window.

> **Temporal stability caveat.** MSR(LW) Sharpe spans 0.41–1.80 across consecutive 5-year
> sub-periods — a range that exceeds the full-sample cross-strategy ranking gap. Part 1 rankings
> describe *family-level tendencies*, not an enduring strategy ordering.

In [ ]:
from IPython.display import display

# ── Top-10 by Sharpe (exclude SHY-concentration artifact) ──────────────────
excl  = "VMP(GMV(sample))"
top10 = (
    canon[canon["strategy"] != excl]
    .nlargest(10, "sharpe")[["display", "family", "sharpe", "max_dd", "turnover", "net_10bps"]]
    .rename(columns={
        "display":   "Strategy",
        "family":    "Family",
        "sharpe":    "Sharpe",
        "max_dd":    "Max DD",
        "turnover":  "Turnover",
        "net_10bps": "Net-10bps SR",
    })
    .reset_index(drop=True)
)
top10.index = range(1, 11)
display(
    top10.style
        .format({"Sharpe": "{:.3f}", "Max DD": "{:.1%}",
                 "Turnover": "{:.3f}", "Net-10bps SR": "{:.3f}"})
        .set_caption(
            "Table 1. Top-10 Part-1 strategies by Sharpe (full sample 2003-2026). "
            "VMP(GMV(sample)) excluded -- degenerate SHY-concentration artifact."
        )
)

# ── Family-median Sharpe bar (base strategies only) ─────────────────────────
# FAMILY_COLORS uses 'TSMOM'; canonical CSV stores family as 'TS Momentum'
color_map = {**FAMILY_COLORS, "TS Momentum": FAMILY_COLORS["TSMOM"]}
ew_sharpe = canon.loc[canon["strategy"] == "EW", "sharpe"].iloc[0]

fam_med = (
    canon[~canon["is_vmp"]]
    .groupby("family")["sharpe"]
    .median()
    .sort_values()
)
colors = [color_map.get(f, "#aaaaaa") for f in fam_med.index]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(fam_med.index, fam_med.values, color=colors, height=0.6)
ax.axvline(ew_sharpe, color="#333333", linestyle="--", lw=1.2,
           label=f"EW benchmark ({ew_sharpe:.3f})")
ax.set_xlabel("Median Sharpe  --  full sample 2003-2026 (base strategies only)")
ax.set_title("Part 1: family-median Sharpe, 31 base strategies")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Headline findings — Part 1

**VMP overlay is a universal Sharpe-improver.** Median lift +0.194; sign-test p ~ 6e-8
(24 / 24 strategies improve). No strategy family is immune to the volatility-targeting boost.

**Ledoit-Wolf shrinkage helps mean-variance families; diversification is near-invariant.**
HRP Memmel test p = 0.506 — shrinkage adds no statistically significant edge for hierarchical
risk parity, which by design avoids matrix inversion.

**The regime rule generalises OOS.** SWITCH(v2a), trained exclusively on 2003–2022, beats the
full-sample SWITCH(LW) by delta = +0.434 Sharpe (z = 2.05, p = 0.040) — a clean OOS validation
of the 8-regime classification framework.

**Ranking headline (genuine results):**
- Best in canonical table (62 strategies): **VMP(MDP(LW)) Sharpe 1.372**
- Best-in-study (OOS regime rule): **VMP(SWITCH(v2a)) Sharpe 1.660**
- Excluded artifact: VMP(GMV(sample)) 1.345 — collapses to near-100% SHY; not a genuine allocation

**Temporal caveat.** MSR(LW) Sharpe spans 0.41–1.80 across consecutive 5-year sub-periods —
a range that exceeds the full-sample cross-strategy ranking gap. Part 1 rankings describe
family-level tendencies, not an enduring strategy ordering.

Paper 2 measures learned approaches against this deterministic bar on the true OOS window
(2023–2026), with epistemic discounting applied by paradigm.

## Repository reading guide

| # | Paper | Title / scope |
|---|---|---|
| [00](00_overview.ipynb) | — | **Overview & Reading Guide** *(this notebook)* |
| [01](01_paper_reproduction.ipynb) | Paper 1 | **Families Comparison** — master 62-strategy table, §3 statistical tests (VMP lift, LW shrinkage, Memmel, SWITCH(v2a)) |
| [02](02_practitioner_analytics.ipynb) | Paper 1 | **Practitioner Analytics** — cumulative wealth, drawdowns, sub-period Sharpe, turnover diagnostics |
| [03](03_ml_strategies.ipynb) | Paper 2 | **ML Strategies** — Lasso / RF / XGBoost return forecasting; headline MSR(Ensemble_mu) OOS Sharpe 2.579 |
| [05](05_dl_portfolio_construction_exploration.ipynb) | Paper 2 | **DL Portfolio Construction** — BSV 2-asset direct-weight walk-forward (JPM-spec); realized-power feature probe |
| [06a](06a_rl_reinforce.ipynb) | Paper 2 | **RL I — REINFORCE** — N=29 simplex-policy, lambda ablation, walk-forward; static-collapse negative result |
| [06b](06b_rl_ppo.ipynb) | Paper 2 | **RL II — PPO** — N=29 proximal policy optimisation, walk-forward |
| [07](07_unsupervised_representation_learning.ipynb) | Paper 2 | **Unsupervised Representation Learning** — PCA factors, correlation geometry, clustering, rolling stability |
| [07b](07b_pca_dislocation_macro_momentum_dashboard.ipynb) | Paper 2 | **PCA Dislocation Dashboard** — tactical dislocation + momentum + macro/regime diagnostic |
| [08](08_llm_bl_views.ipynb) | Paper 2 | **LLM-Augmented BL Views** — 5 generator variants OOS 2023–2026; hindsight test for training-cutoff contamination |
| 09 *(planned)* | Paper 2 | **Learned Synthesis** — closure comparison across all learned paradigms |
| 99 *(planned)* | Paper 1 | **Reproducibility Guard** — 62-row assertion regression test |

Notebooks `04` / `04b` are DL signal-strategy experiments (M4 CPU + RTX Blackwell CUDA runs)
supporting the Paper 2 DL track; retained for full reproducibility but not primary reading-guide entries.